###Data Reading

In [0]:
dbutils.fs.ls("/Volumes/workspace/raw_gcd/contracts_file/")

In [0]:
df_sales=spark.read.csv("/Volumes/workspace/raw_gcd/contracts_file/BigMart Sales.csv", header=True, inferSchema=True)

In [0]:
df_sales.show(10)

In [0]:
df_sales.display(10)

In [0]:
df_drivers= spark.read.format('json').option('header',True).option('inferSchema',True).load("/Volumes/workspace/raw_gcd/contracts_file/drivers.json")

In [0]:
df_drivers.display(10)

###Schema Definition

In [0]:
df_sales.printSchema()

###DDL Schema

In [0]:
my_ddl_schema = '''
Item_Identifier STRING,
Item_Weight STRING,
Item_Fat_Content STRING,
Item_Visibility DOUBLE,
Item_Type STRING,
Item_MRP DOUBLE,
Outlet_Identifier STRING,
Outlet_Establishment_Year INT,
Outlet_Size STRING,
Outlet_Location_Type STRING,
Outlet_Type STRING,
Item_Outlet_Sales DOUBLE
'''

In [0]:
df_sales=spark.read.format('csv').option('header',True).schema(my_ddl_schema).load("/Volumes/workspace/raw_gcd/contracts_file/BigMart Sales.csv")


In [0]:
df_sales.display()

In [0]:
df_sales.printSchema()

###StructType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
df_drivers.printSchema()

#####TRANSFORMATIONS

###SELECT

In [0]:
df_sales.display()

In [0]:
df_sales.select('Item_Identifier','Item_Weight','Item_Fat_Content').display()

In [0]:
df_sales.select(col("Item_Identifier"),col("Item_Weight"),col("Item_Fat_Content")).display()

###ALIAS

In [0]:
df_sales.select(col('Item_Identifier').alias('Item_ID')).display()

In [0]:
df_sales.select(col("Item_MRP").alias('MRP')).display()

###FILTER/WHERE
row level slicing

In [0]:
df_sales.display()

####scenario-1: Filter the data with Fat Content Regular

In [0]:
df_sales.select(col("Item_Fat_Content")).distinct().display()

In [0]:
df_sales.filter(col("Item_Fat_Content")=='Regular').display()

###scenario-2: Slice the data with Item_Type Soft Drinks and weight less than 10

In [0]:
df_sales.filter((col('Item_Type')=='Soft Drinks') & (col('Item_Weight')<10)).display()

##scenario-3: fetch the data with tier in (tier1,tier2) and outlet size is NULL

In [0]:
df_sales.filter(col("Outlet_Size").isNull() & col("Outlet_Location_Type").isin('Tier 1','Tier 2')).display()

###withColumnRenamed

In [0]:
df_sales.withColumnRenamed("Item_Weight","Item_Wt").display()

###withColumn

###scenario-1: create a new column called flag with 'new' as values

In [0]:
df_sales=df_sales.withColumn('flag',lit('new'))

In [0]:
df_sales.display()

###create a new column with product of Item_Weight and Item_MRP

In [0]:
df_sales=df_sales.withColumn('product',col("Item_Weight")*col("Item_MRP"))

In [0]:
df_sales.display()

###scenario-3: modify the Item_Fat_Content values

In [0]:
df_sales.withColumn('Item_Fat_Content',regexp_replace(col("Item_Fat_Content"),'Regular','Reg'))\
        .withColumn('Item_Fat_Content',regexp_replace(col("Item_Fat_Content"),'Low Fat','Lf')).display()

###Type Casting

In [0]:
df_sales= df_sales.withColumn('Item_Weight',col("Item_Weight").cast(StringType()))

In [0]:
df_sales.printSchema()

###sort/orderby

In [0]:
df_sales.sort(col("Item_Weight").desc()).display()

In [0]:
df_sales.sort(col("Item_MRP").asc()).display()

In [0]:
df_sales.sort(['Item_Weight','Item_Visibility'], ascending=[0,0]).display()

In [0]:
df_sales.sort(["Item_Weight","Item_MRP"], ascending=[0,1]).display()

###LIMIT

In [0]:
df_sales.limit(10).display()

###INTERMEDIATE

###DROP

In [0]:
df_sales.drop("Item_Visibility").display()

In [0]:
df_sales.drop("Item_Visibility","Item_Type").display()

In [0]:
df_sales.display()

###dropDuplicates

scenario1

In [0]:
df_sales.dropDuplicates().display()

In [0]:
df_sales.distinct().display()

scenario2

In [0]:
df_sales.dropDuplicates(subset=["Outlet_Type"]).display()

###UNION

Preparing DataFrames

In [0]:
data1=[('1','kad'),
       ('2','sid')
       ] 
schema1='id STRING, name STRING'

df1=spark.createDataFrame(data1,schema1)   

In [0]:
data2=[('3','rahul'),
       ('4','jas')
       ]
schema2='id STRING, name STRING'

df2=spark.createDataFrame(data2,schema2)

In [0]:
display(df1)

In [0]:
display(df2)

In [0]:
df1.union(df2).display()

###Union by Name

In [0]:
data1=[('kad','1'),
       ('sid','2')
       ] 
schema1='name STRING, id STRING' 

df1=spark.createDataFrame(data1,schema1)   

In [0]:
df1.union(df2).display()

In [0]:
df1.unionByName(df2).display()

###STRING FUNCTIONS

In [0]:
df_sales.limit(100).distinct().display()

###initcap(), lower(), upper()

In [0]:
df_sales.select(upper("Item_Type").alias('Item_Type')).display()

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

###DATE FUNCTIONS

CURRENT_DATE()\
DATE_ADD()\
DATE_SUB()

In [0]:
df_sales = df_sales.withColumn('curr_date',current_date())
df_sales.display()

In [0]:
df_sales=df_sales.withColumn('week_after',date_add('curr_date',7))
df_sales.display()

In [0]:
df_sales.withColumn('prev_week',date_add('curr_date',-7)).display()

In [0]:
df_sales.withColumn('prev_week',date_sub('curr_date',7)).display()

###DateDIFF

In [0]:
df_sales.withColumn('datediff', datediff('week_after','curr_date')).display()

###date_format()

In [0]:
df_sales.withColumn('week_after',date_format('week_after','dd/MM/yyyy')).display()

###HANDLING NULLS

#####dropna()

In [0]:
df_sales.dropna('all').display()

In [0]:
df_sales.dropna('any').display()

In [0]:
df_sales.dropna(subset="Outlet_Size").display()

######Filling NULLS  fillna()

In [0]:
df_sales.display()

In [0]:
df_sales.fillna('NA').display()

In [0]:
df_sales.fillna(0).display()

In [0]:
df_sales.fillna(2000,subset=["Item_Weight"]).display()

####SPLIT and Indexing

In [0]:
df_sales.withColumn('Outlet_Type',split('Outlet_Type',' ')[1]).display()

###Explode,  new row for every value in the list

In [0]:
df_exp=df_sales.withColumn('Outlet_Type',split('Outlet_Type',' '))

In [0]:
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()

####ARRAY_CONTAINS()

In [0]:
df_exp.withColumn('type2_flag', array_contains('Outlet_Type','Type2')).display()

#####GROUP BY 

#####find the sum of MRP for each Item Type

In [0]:
df_sales.groupBy("Item_Type").agg(sum("Item_MRP").alias('MRP')).display()

In [0]:
####avg

df_sales.groupBy('Item_Type').agg(avg("Item_MRP").alias('avg')).display()

In [0]:
df_sales.groupBy("Item_Type").min("Item_MRP").display()

In [0]:
df_sales.groupBy("Item_Type").max('Item_MRP').display()

#####scenario-3

In [0]:
df_sales.groupBy("Item_Type","Outlet_Size",).agg(sum("Item_MRP").alias('Total_MRP')).display()

In [0]:
df_sales.groupBy("Item_Type","Outlet_Size").agg(sum("Item_MRP"),avg("Item_MRP")).display()